# Crop Yield Forecasting

## Project Overview

Forecasts **daily crop yield proxy** (tons harvested) over a 14-day horizon. Synthetic data spans 2 years with strong annual seasonality (growing/harvest seasons), weather variability, and a slight productivity trend.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily harvest output, predict the next 14 days. Crop yield forecasts support food supply planning, commodity pricing, logistics, and agricultural policy decisions.

## Why This Project Matters

Global food security depends on accurate crop yield estimation. Governments, commodity traders, insurance providers, and logistics companies all rely on yield forecasts. Even a 5% improvement in forecast accuracy can prevent significant economic losses and food waste.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily yield proxy
- Strong annual cycle (planting → growth → harvest → dormancy)
- Weather-driven daily variability
- Slight upward productivity trend (better farming practices)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `yield_tons` | Daily harvested yield (tons) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'yield_tons'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Strong annual cycle: summer harvest peak
seasonal = 50 + 40 * np.sin(2 * np.pi * (t - 100) / 365)
seasonal = np.maximum(seasonal, 5)  # minimum dormancy yield

# Productivity trend
trend = 0.01 * t

# Weather variability (autocorrelated)
weather = np.zeros(N_POINTS)
weather[0] = np.random.normal(0, 3)
for i in range(1, N_POINTS):
    weather[i] = 0.5 * weather[i-1] + np.random.normal(0, 3)

# Occasional bad weather events (drought/frost)
bad_events = np.where(np.random.random(N_POINTS) < 0.03, -np.random.uniform(10, 25, N_POINTS), 0)

yield_tons = seasonal + trend + weather + bad_events
yield_tons = np.maximum(yield_tons, 1).round(1)

df = pd.DataFrame({'date': dates, 'yield_tons': yield_tons})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,yield_tons
0,2022-01-01,1.0
1,2022-01-02,10.7
2,2022-01-03,12.4
3,2022-01-04,15.8
4,2022-01-05,12.3


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('yield_tons Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("yield_tons Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

yield_tons Statistics:
count    730.000000
mean      53.123699
std       28.715119
min        1.000000
25%       26.525000
50%       51.850000
75%       80.900000
max      102.900000
Name: yield_tons, dtype: float64

CV: 0.541
Skewness: 0.001


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        9.5 | RMSE:        9.9 | MAPE: 56.76%
Seasonal Naive (7)             | MAE:        3.2 | RMSE:        4.2 | MAPE: 20.20%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared       RMSE  Time Taken
Model                                                                            
KernelRidge                        6331.424292 -485.955715  57.582731    0.015499
QuantileRegressor                  2423.926480 -185.378960  35.624269    0.043476
DummyRegressor                     2322.899767 -177.607674  34.873664    0.005582
GaussianProcessRegressor            139.537815   -9.656755   8.518439    0.036360
TweedieRegressor                     71.896378   -4.453568   6.093795    0.009000
GammaRegressor                       64.878540   -3.913734   5.784334    2.539024
SVR                                  59.169470   -3.474575   5.519801    0.019346
NuSVR                                55.584489   -3.198807   5.347004    0.016946
ElasticNet                           49.585412   -2.737339   5.044624    0.006859
PassiveAggressiveRegressor           48.834397   -2.679569   5.005483    0.00

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        2.9 | RMSE:        3.4 | MAPE: 14.42%
FLAML Test (catboost)          | MAE:        2.3 | RMSE:        3.3 | MAPE: 15.15%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        4.4 | RMSE:        5.1 | MAPE: 27.67%
SF AutoETS                     | MAE:        1.6 | RMSE:        2.3 | MAPE: 10.14%
SF SeasonalNaive               | MAE:        4.5 | RMSE:        5.4 | MAPE: 28.16%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE     RMSE      MAPE
           SF AutoETS 1.597118 2.253116 10.141906
FLAML Test (catboost) 2.293499 3.275985 15.148810
     FLAML (catboost) 2.939394 3.380732 14.421799
   Seasonal Naive (7) 3.192857 4.183727 20.196424
         SF AutoARIMA 4.440957 5.128494 27.666277
     SF SeasonalNaive 4.528571 5.394839 28.162107
   Naive (Last Value) 9.542857 9.886788 56.764690

Top 3:
                model      MAE     RMSE      MAPE
           SF AutoETS 1.597118 2.253116 10.141906
FLAML Test (catboost) 2.293499 3.275985 15.148810
     FLAML (catboost) 2.939394 3.380732 14.421799


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -2.28, Std: 2.35


## Interpretation and Business Insight

- Crop yield follows a strong annual cycle driven by growing seasons
- Summer/autumn harvest periods show peak output
- Winter/early spring is dormancy with minimal yield
- Weather events (drought, frost) cause sudden drops
- Slight upward trend reflects improved farming practices

## Limitations

1. Synthetic — real yields depend on soil, irrigation, pest pressure, variety
2. Daily granularity is unusual; real yield data is typically seasonal/annual
3. No weather features (temperature, rainfall, soil moisture)
4. Single crop type — real farms grow multiple crops
5. No spatial variation (single field/region)

## How to Improve This Project

1. Add weather features (temperature, rainfall, solar radiation)
2. Use satellite vegetation indices (NDVI) as predictors
3. Model at seasonal/annual granularity with panel data
4. Include soil type and irrigation data
5. Apply crop simulation models (DSSAT, APSIM) alongside ML

## Production Considerations

- Seasonal yield forecasting for commodity markets
- Integration with satellite remote sensing
- Insurance loss estimation and risk pricing
- Supply chain and logistics planning

## Common Mistakes

1. Confusing daily proxy with actual seasonal yield totals
2. Ignoring the strong annual cycle when selecting models
3. Not accounting for weather as a key driver
4. Using stationary models on highly seasonal data
5. Overfitting to noise in daily variability

## Mini Challenge / Exercises

1. Aggregate to weekly yield and compare forecast accuracy
2. Detect drought/frost events using anomaly detection
3. Build a seasonal yield total forecast (annual)
4. Compare harvest-season-only vs full-year model performance

## Final Summary / Key Takeaways

1. Crop yield has the strongest annual seasonality among common forecasting targets
2. Weather events are the primary source of forecast difficulty
3. Real deployment needs satellite and weather data integration
4. Statistical models capture the seasonal pattern well
5. ML models add value through weather-feature lag interactions

In [18]:
import json
metrics = {
    'project': 'Crop Yield Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Crop Yield Forecasting — Complete ===")

Metrics saved.

=== Crop Yield Forecasting — Complete ===
